# 卷积神经网络与图像分类

卷积层保留图像中像素的邻接关系，并在不同位置使用同一组滤波器。本节先用简单 CNN 识别 MNIST 数字，再用 ResNet18 分类 CIFAR-10 彩色图像。

随后在各自的数据集内比较 MLP、CNN 和通道数减半的 CNN，观察准确率与计算开销，并查看卷积层的特征图。

[课程目录与安装说明](../../README.md)


In [ ]:
from biai.paths import DATA_DIR
from biai.reproducibility import seed_everything

SEED = 0
seed_everything(SEED, deterministic=True)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import random
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
import time


## 准备 MNIST

MNIST 图像只有一个灰度通道。代码先将像素转为 [0, 1] 的浮点数，再按给定的均值和标准差做标准化。MNIST 和后面使用的 CIFAR-10 都会下载到仓库的 `data/`，之后直接读取本地缓存。


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize grayscale pixel values.
])

train_dataset = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

indices = torch.randperm(len(train_dataset), generator=torch.Generator().manual_seed(SEED))
validation_size = len(train_dataset) // 10
val_dataset = Subset(train_dataset, indices[:validation_size].tolist())
train_dataset = Subset(train_dataset, indices[validation_size:].tolist())
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=1000, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)

In [ ]:
print(f"数据集大小: {len(train_dataset)}")
print(f"图像形状: {train_dataset[0][0].shape}")


In [ ]:
idx = random.sample(range(len(train_dataset)), 1)
image, target = train_dataset[idx[0]]
print(f"target: {target}")

# Convert from CHW to HWC for plotting.
image_np = image.numpy().transpose(1, 2, 0)

fig, ax1 = plt.subplots(1, 1, figsize=(12, 5))
ax1.imshow(image_np.squeeze(), cmap='gray')


## 两个卷积块的分类器

每个卷积块先用 3 × 3 卷积和 ReLU 提取特征，再用最大池化将长宽减半。输入依次从 28 × 28 变为 14 × 14、7 × 7，最后将 64 张特征图展平并输出 10 个类别分数。

模型使用 Adam 训练 5 轮。从原训练集固定留出 10% 作为验证集，观察每轮训练与验证的交叉熵、准确率；测试集只在训练结束后评估一次。

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, input_channels=1, image_size=28, width=32):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, width, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(width, width * 2, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(width * 2 * (image_size // 4) ** 2, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)  # 28 x 28 -> 14 x 14.
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)  # 14 x 14 -> 7 x 7.
        return self.fc1(torch.flatten(x, 1))


seed_everything(SEED, deterministic=True)
cnn_model = SimpleCNN().to(device)
print(cnn_model)

In [ ]:
criterion = nn.CrossEntropyLoss()


def train(model, loader, optimizer, epoch):
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * target.numel()
        correct += (output.argmax(1) == target).sum().item()
        count += target.numel()
    return loss_sum / count, correct / count


def test(model, loader):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss_sum += F.cross_entropy(output, target, reduction="sum").item()
            correct += (output.argmax(1) == target).sum().item()
            count += target.numel()
    return loss_sum / count, correct / count


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for split in ("train", "validation"):
        axes[0].plot([row[f"{split}_loss"] for row in history], label=split)
        axes[1].plot([row[f"{split}_accuracy"] for row in history], label=split)
    axes[0].set_ylabel("Cross entropy")
    axes[1].set_ylabel("Accuracy")
    for ax in axes:
        ax.set_xlabel("Epoch (from 0)")
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def fit_image_classifier(model, loaders, epochs, learning_rate, optimizer_name):
    train_loader, validation_loader, test_loader = loaders
    seed_everything(SEED, deterministic=True)
    train_loader.generator.manual_seed(SEED)
    model = model.to(device)
    if optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        scheduler = None
    else:
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history, training_seconds = [], 0.0
    for epoch in range(epochs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        train_loss, train_acc = train(model, train_loader, optimizer, epoch)
        if device.type == "cuda":
            torch.cuda.synchronize()
        training_seconds += time.perf_counter() - start
        val_loss, val_acc = test(model, validation_loader)
        history.append(dict(train_loss=train_loss, train_accuracy=train_acc,
                            validation_loss=val_loss, validation_accuracy=val_acc))
        print(f"Epoch {epoch+1}: train={train_acc:.2%}, validation={val_acc:.2%}")
        if scheduler is not None:
            scheduler.step()
    test_loss, test_acc = test(model, test_loader)
    result = dict(history=history, test_accuracy=test_acc, test_loss=test_loss,
                  parameters=sum(p.numel() for p in model.parameters()),
                  training_seconds=training_seconds,
                  examples_per_second=epochs * len(train_loader.dataset) / training_seconds)
    plot_history(history, type(model).__name__)
    return result


MNIST_EPOCHS = 5
mnist_loaders = (train_loader, val_loader, test_loader)
mnist_cnn_result = fit_image_classifier(cnn_model, mnist_loaders, MNIST_EPOCHS, 0.001, "adam")
cnn_history = mnist_cnn_result["history"]
cnn_test_accuracy = mnist_cnn_result["test_accuracy"]
print(f"Final MNIST test: {cnn_test_accuracy:.2%}")


## 在 CIFAR-10 上训练 ResNet18

CIFAR-10 包含 32 × 32 的彩色图像。这里使用更深的 ResNet18，它通过残差连接将卷积块的输入与卷积结果相加；尺寸不一致时，先对输入作变换再相加。

为适应小图像，代码将输入卷积改为步长 1 的 3 × 3 卷积，去掉紧随其后的最大池化，以保留空间细节；最后的分类层输出 10 个分数。`weights=None` 表示从随机参数开始训练。

训练样本经过随机裁剪和水平翻转，再做像素标准化；验证与测试样本只做标准化。从原训练集固定留出 10% 用于验证，采用带动量的 SGD 训练 20 轮，学习率按余弦曲线逐步降低。

每轮结束后评估验证集，全部训练结束后评估测试集。训练曲线汇总一轮中各批次的表现，验证曲线使用该轮结束时的参数，两者并不是在同一组参数上计算的。


In [ ]:
# Apply augmentation to training images only.
transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_cifar_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    *transform_cifar.transforms
])
train_dataset_cifar = datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=transform_cifar_train
)
validation_source_cifar = datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=transform_cifar
)
test_dataset_cifar = datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=transform_cifar
)
indices = torch.randperm(len(train_dataset_cifar), generator=torch.Generator().manual_seed(SEED))
validation_size = len(train_dataset_cifar) // 10
val_dataset_cifar = Subset(validation_source_cifar, indices[:validation_size].tolist())
train_dataset_cifar = Subset(train_dataset_cifar, indices[validation_size:].tolist())
train_loader_cifar = DataLoader(train_dataset_cifar, batch_size=128, shuffle=True,
                                generator=torch.Generator().manual_seed(SEED))
val_loader_cifar = DataLoader(val_dataset_cifar, batch_size=100, shuffle=False)
test_loader_cifar = DataLoader(test_dataset_cifar, batch_size=100, shuffle=False)

In [ ]:
print(f"数据集大小: {len(train_dataset_cifar)}")
print(f"图像形状: {train_dataset_cifar[0][0].shape}")
idx = random.sample(range(len(train_dataset_cifar)), 1)
image, target = train_dataset_cifar[idx[0]]
print(f"target: {target}")

# Convert from CHW to HWC for plotting.
image_np = image.numpy().transpose(1, 2, 0)
image_np = (image_np * (0.2470, 0.2435, 0.2616) + (0.4914, 0.4822, 0.4465)).clip(0, 1)

fig, ax1 = plt.subplots(1, 1, figsize=(12, 5))
ax1.imshow(image_np.squeeze(), cmap='gray')

In [ ]:
# Initialize ResNet18 without pretrained weights.
seed_everything(SEED, deterministic=True)
resnet18 = models.resnet18(weights=None)
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet18.maxpool = nn.Identity()
num_ftrs = resnet18.fc.in_features
print(f"Feature dmension: {num_ftrs}")
resnet18.fc = nn.Linear(num_ftrs, 10)  # Replace the classifier with 10 outputs.

resnet18 = resnet18.to(device)

In [ ]:
initial_val_loss, initial_val_accuracy = test(resnet18, val_loader_cifar)
print(f"Before training: validation loss={initial_val_loss:.4f}, accuracy={initial_val_accuracy:.2%}")

In [ ]:
CIFAR_EPOCHS = 20
cifar_loaders = (train_loader_cifar, val_loader_cifar, test_loader_cifar)
cifar_resnet_result = fit_image_classifier(resnet18, cifar_loaders, CIFAR_EPOCHS, 0.1, "sgd")
cifar_history = cifar_resnet_result["history"]
cifar_test_accuracy = cifar_resnet_result["test_accuracy"]
print(f"Final CIFAR test: {cifar_test_accuracy:.2%}")

## 比较 MLP、CNN 与窄 CNN

前面已经得到 MNIST 上的 CNN 和 CIFAR-10 上的 ResNet18。下面继续训练其余模型，将同一数据集上的结果放在一起比较。窄 CNN 将两个卷积层的通道数都减半，用来观察减少参数后能保留多少分类能力。

| 数据集 | 比较的模型 | 优化器与初始学习率 | 训练轮数 |
| --- | --- | --- | --- |
| MNIST | MLP、CNN、窄 CNN | Adam，0.001 | 5 |
| CIFAR-10 | MLP、CNN、窄 CNN、ResNet18 | 带动量 SGD，0.1，余弦衰减 | 20 |

同一数据集内沿用相同的数据划分、预处理、训练增强和批次顺序。所有模型从随机参数开始训练，随机种子相同；由于网络结构不同，参数无法逐个对应。本页 MLP 的设置也与前一份 notebook 不同，应在本页对应的数据集内比较。

表格中的训练时间包含读取数据和参数更新，不包含验证、测试与绘图；GPU 计时前后进行同步。准确率和速度都对应当前设备与训练设置，各结构尚未分别调到最优。


In [ ]:
class ImageMLP(nn.Module):
    def __init__(self, input_channels, image_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(), nn.Linear(input_channels * image_size**2, 256),
            nn.ReLU(), nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.layers(x)


image_results = {("MNIST", "CNN"): mnist_cnn_result,
                 ("CIFAR-10", "ResNet18"): cifar_resnet_result}
comparison_models = {}
for dataset_name, channels, size, loaders, budget, lr, optimizer_name in (
    ("MNIST", 1, 28, mnist_loaders, MNIST_EPOCHS, 0.001, "adam"),
    ("CIFAR-10", 3, 32, cifar_loaders, CIFAR_EPOCHS, 0.1, "sgd"),
):
    names = ("MLP", "Narrow CNN") if channels == 1 else ("MLP", "CNN", "Narrow CNN")
    for name in names:
        seed_everything(SEED, deterministic=True)
        comparison_model = (ImageMLP(channels, size) if name == "MLP" else
                            SimpleCNN(channels, size, width=16 if name == "Narrow CNN" else 32))
        image_results[(dataset_name, name)] = fit_image_classifier(
            comparison_model, loaders, budget, lr, optimizer_name
        )
        comparison_models[(dataset_name, name)] = comparison_model

print(f"{'Dataset / model':28s} {'Val':>8s} {'Test':>8s} {'Params':>10s} {'Train s':>10s} {'Images/s':>10s}")
for (dataset_name, name), row in image_results.items():
    print(f"{dataset_name + ' / ' + name:28s} {row['history'][-1]['validation_accuracy']:8.2%} "
          f"{row['test_accuracy']:8.2%} {row['parameters']:10d} "
          f"{row['training_seconds']:10.1f} {row['examples_per_second']:10.1f}")


## 观察卷积特征图

下面取验证集的第一张图，显示输入图像，以及两层卷积经过 ReLU 后的前八个通道。第一层在原图尺寸上提取特征；第二层接收经过一次池化的特征，因此空间尺寸更小。

可以观察哪些位置有较强响应，再换几张图看这些响应是否重复出现。单张图上的响应还不足以确定一个通道识别了什么。各特征图单独缩放颜色，颜色深浅只能在同一张特征图内比较。


In [ ]:
def show_feature_maps(model, dataset, mean, std, title):
    was_training = model.training
    model.eval()
    image, label = dataset[0]
    with torch.no_grad():
        x = image[None].to(device)
        first = F.relu(model.conv1(x))
        second = F.relu(model.conv2(F.max_pool2d(first, 2)))
    model.train(was_training)
    display_image = (image * torch.tensor(std)[:, None, None]
                     + torch.tensor(mean)[:, None, None]).clamp(0, 1)
    fig, axes = plt.subplots(3, 8, figsize=(14, 5))
    for ax in axes.flat:
        ax.axis("off")
    axes[0, 0].imshow(display_image.permute(1, 2, 0).squeeze().numpy(), cmap="gray")
    axes[0, 0].set_title(f"Input: {label}")
    for row, features in enumerate((first, second), 1):
        for channel in range(8):
            axes[row, channel].imshow(features[0, channel].cpu().numpy(), cmap="viridis")
            axes[row, channel].set_title(f"Layer {row}, ch {channel}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_feature_maps(cnn_model, val_dataset, (0.1307,), (0.3081,), "MNIST features")
show_feature_maps(comparison_models[("CIFAR-10", "CNN")], val_dataset_cifar,
                  (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616), "CIFAR-10 features")


## 可选扩展

- 可以调整卷积通道数、卷积块数，或尝试全局平均池化，观察准确率、参数量和训练时间的变化。每次改变一个因素，使用验证集选择设置。
- 可以换几类图像观察两层特征图，看看同一个通道对不同输入的响应是否相似。
